# Chapter 1.3: Classical Collaborative Filtering

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement User-based CF with cosine and Pearson similarity
2. Implement Item-based CF and understand its computational advantages
3. Build a Matrix Factorization model using SGD and ALS
4. Understand and implement BPR (Bayesian Personalized Ranking) loss
5. Compare the tradeoffs between memory-based and model-based CF
6. Evaluate CF methods on synthetic data using ranking metrics
7. Identify failure modes and limitations of classical CF

## Prerequisites

- Chapter 1.1: The Recommendation Problem (evaluation metrics)
- Chapter 1.2: User-Item Interaction Data (matrices, splitting)
- Basic PyTorch (tensors, autograd, optimizers)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part1/chapter_1.3_collaborative_filtering.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part1/chapter_1.3_collaborative_filtering.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.metrics import roc_auc_score

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

print(f"PyTorch version: {torch.__version__}")
print("All imports successful!")

In [ ]:
# Generate synthetic data (same generator as Chapter 1.2)
def generate_synthetic_data(n_users=300, n_items=100, density=0.08, seed=42):
    np.random.seed(seed)
    
    # Latent factors for users and items (ground truth)
    k = 5
    user_factors = np.random.randn(n_users, k) * 0.5
    item_factors = np.random.randn(n_items, k) * 0.5
    user_bias = np.random.randn(n_users) * 0.3
    item_bias = np.random.randn(n_items) * 0.3
    
    # True preference scores
    true_scores = 3.5 + user_factors @ item_factors.T + user_bias[:, None] + item_bias[None, :]
    true_scores = np.clip(true_scores, 1, 5)
    
    # Sample observed entries
    item_popularity = np.random.power(0.3, n_items)
    item_popularity /= item_popularity.sum()
    
    rows, cols, ratings = [], [], []
    for u in range(n_users):
        n_ratings = max(1, int(np.random.exponential(density * n_items)))
        n_ratings = min(n_ratings, n_items)
        items = np.random.choice(n_items, n_ratings, replace=False, p=item_popularity)
        for i in items:
            rows.append(u)
            cols.append(i)
            r = true_scores[u, i] + np.random.randn() * 0.5
            ratings.append(np.clip(round(r * 2) / 2, 1, 5))
    
    R = sparse.csr_matrix((ratings, (rows, cols)), shape=(n_users, n_items))
    return R, true_scores


R, true_scores = generate_synthetic_data()
R_dense = R.toarray()
n_users, n_items = R.shape

print(f"Interaction matrix: {n_users} users x {n_items} items")
print(f"Observed ratings: {R.nnz}")
print(f"Density: {R.nnz / (n_users * n_items):.2%}")

## 1. User-Based Collaborative Filtering

### Core Idea
Find users similar to the target user, then predict ratings using their opinions.

### Similarity Measures

**Cosine Similarity:**
$$
\text{sim}(u, v) = \frac{\mathbf{r}_u \cdot \mathbf{r}_v}{\|\mathbf{r}_u\| \cdot \|\mathbf{r}_v\|}
$$

**Pearson Correlation** (centered cosine):
$$
\text{sim}(u, v) = \frac{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_{i \in I_{uv}} (r_{ui} - \bar{r}_u)^2} \sqrt{\sum_{i \in I_{uv}} (r_{vi} - \bar{r}_v)^2}}
$$

### Prediction
$$
\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v \in N_k(u)} \text{sim}(u, v) \cdot (r_{vi} - \bar{r}_v)}{\sum_{v \in N_k(u)} |\text{sim}(u, v)|}
$$

where $N_k(u)$ is the set of $k$ most similar users who rated item $i$.

In [ ]:
class UserBasedCF:
    def __init__(self, k=20, similarity='cosine'):
        self.k = k
        self.similarity = similarity
    
    def fit(self, R):
        """Compute user-user similarity matrix."""
        self.R = R.copy()
        self.n_users, self.n_items = R.shape
        self.user_means = np.zeros(self.n_users)
        
        # Compute mean rating per user (only over observed entries)
        for u in range(self.n_users):
            rated = R[u].nonzero()[1]
            if len(rated) > 0:
                self.user_means[u] = R[u, rated].toarray().mean()
        
        if self.similarity == 'cosine':
            self.sim_matrix = self._cosine_similarity(R)
        elif self.similarity == 'pearson':
            self.sim_matrix = self._pearson_similarity(R)
        
        # Zero out self-similarity
        np.fill_diagonal(self.sim_matrix, 0)
        return self
    
    def _cosine_similarity(self, R):
        R_dense = R.toarray()
        norms = np.linalg.norm(R_dense, axis=1, keepdims=True)
        norms[norms == 0] = 1
        R_norm = R_dense / norms
        return R_norm @ R_norm.T
    
    def _pearson_similarity(self, R):
        R_dense = R.toarray()
        # Center by user mean (only for observed entries)
        R_centered = R_dense.copy()
        for u in range(self.n_users):
            mask = R_dense[u] != 0
            R_centered[u, mask] -= self.user_means[u]
            R_centered[u, ~mask] = 0  # Keep unobserved as 0
        
        norms = np.linalg.norm(R_centered, axis=1, keepdims=True)
        norms[norms == 0] = 1
        R_norm = R_centered / norms
        return R_norm @ R_norm.T
    
    def predict(self, user_id, item_id):
        """Predict rating for (user, item)."""
        # Find top-k similar users who rated this item
        sims = self.sim_matrix[user_id].copy()
        item_raters = self.R[:, item_id].toarray().flatten()
        rated_mask = item_raters != 0
        sims[~rated_mask] = 0  # Ignore users who haven't rated the item
        
        # Top-k neighbors
        top_k_idx = np.argsort(sims)[-self.k:]
        top_k_sims = sims[top_k_idx]
        
        if np.sum(np.abs(top_k_sims)) == 0:
            return self.user_means[user_id]
        
        # Weighted average of deviations from mean
        deviations = item_raters[top_k_idx] - self.user_means[top_k_idx]
        prediction = self.user_means[user_id] + (
            np.sum(top_k_sims * deviations) / np.sum(np.abs(top_k_sims))
        )
        return np.clip(prediction, 1, 5)
    
    def predict_all(self, user_id):
        """Predict ratings for all items for a user."""
        return np.array([self.predict(user_id, i) for i in range(self.n_items)])


# Train UserCF
user_cf_cosine = UserBasedCF(k=20, similarity='cosine').fit(R)
user_cf_pearson = UserBasedCF(k=20, similarity='pearson').fit(R)

# Test on a sample user
test_user = 0
predictions_cosine = user_cf_cosine.predict_all(test_user)
predictions_pearson = user_cf_pearson.predict_all(test_user)

# Compare with true scores
rated_items = R[test_user].nonzero()[1]
unrated_items = np.setdiff1d(np.arange(n_items), rated_items)

print(f"User {test_user}: {len(rated_items)} rated items")
print(f"\nSample predictions (unrated items):")
for i in unrated_items[:5]:
    print(f"  Item {i}: True={true_scores[test_user, i]:.2f}, "
          f"Cosine={predictions_cosine[i]:.2f}, Pearson={predictions_pearson[i]:.2f}")

## 2. Item-Based Collaborative Filtering

Instead of user-user similarity, compute **item-item** similarity.

$$
\hat{r}_{ui} = \frac{\sum_{j \in N_k(i) \cap I_u} \text{sim}(i, j) \cdot r_{uj}}{\sum_{j \in N_k(i) \cap I_u} |\text{sim}(i, j)|}
$$

> **💡 Concept:** ItemCF is often preferred over UserCF in practice because:
> 1. Items are more "stable" than users (a movie doesn't change; user tastes do)
> 2. Item similarity can be precomputed and cached
> 3. Better scalability when users >> items

In [ ]:
class ItemBasedCF:
    def __init__(self, k=20):
        self.k = k
    
    def fit(self, R):
        """Compute item-item similarity matrix."""
        self.R = R.copy()
        self.n_users, self.n_items = R.shape
        
        # Cosine similarity between items (columns)
        R_dense = R.toarray()
        norms = np.linalg.norm(R_dense, axis=0, keepdims=True)
        norms[norms == 0] = 1
        R_norm = R_dense / norms
        self.item_sim = R_norm.T @ R_norm
        np.fill_diagonal(self.item_sim, 0)
        return self
    
    def predict(self, user_id, item_id):
        """Predict rating for (user, item)."""
        user_ratings = self.R[user_id].toarray().flatten()
        rated_mask = user_ratings != 0
        
        sims = self.item_sim[item_id].copy()
        sims[~rated_mask] = 0
        
        top_k_idx = np.argsort(sims)[-self.k:]
        top_k_sims = sims[top_k_idx]
        
        if np.sum(np.abs(top_k_sims)) == 0:
            return user_ratings[rated_mask].mean() if rated_mask.sum() > 0 else 3.0
        
        prediction = np.sum(top_k_sims * user_ratings[top_k_idx]) / np.sum(np.abs(top_k_sims))
        return np.clip(prediction, 1, 5)
    
    def predict_all(self, user_id):
        return np.array([self.predict(user_id, i) for i in range(self.n_items)])


item_cf = ItemBasedCF(k=20).fit(R)
predictions_itemcf = item_cf.predict_all(test_user)

print(f"Sample predictions for User {test_user}:")
for i in unrated_items[:5]:
    print(f"  Item {i}: True={true_scores[test_user, i]:.2f}, "
          f"UserCF={predictions_cosine[i]:.2f}, ItemCF={predictions_itemcf[i]:.2f}")

## 3. Matrix Factorization

Factor the interaction matrix into low-rank user and item matrices:

$$
R \approx P Q^T
$$

where $P \in \mathbb{R}^{|U| \times k}$ and $Q \in \mathbb{R}^{|I| \times k}$.

### With Biases
$$
\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^T \mathbf{q}_i
$$

### SGD Optimization
$$
\mathcal{L} = \sum_{(u,i) \in \mathcal{O}} (r_{ui} - \hat{r}_{ui})^2 + \lambda(\|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2 + b_u^2 + b_i^2)
$$

> **🔑 Pro Tip:** Matrix Factorization (especially SVD++) won the Netflix Prize and was the dominant approach from 2006-2015. Understanding MF deeply is essential because many modern deep learning methods are generalizations of MF.

In [ ]:
class MatrixFactorization:
    def __init__(self, n_users, n_items, n_factors=20, lr=0.01, reg=0.02):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        
        # Initialize parameters
        self.P = np.random.randn(n_users, n_factors) * 0.1
        self.Q = np.random.randn(n_items, n_factors) * 0.1
        self.b_u = np.zeros(n_users)
        self.b_i = np.zeros(n_items)
        self.mu = 0.0
    
    def fit(self, R, n_epochs=50, verbose=True):
        """Train with SGD."""
        # Extract observed entries
        rows, cols = R.nonzero()
        values = np.array(R[rows, cols]).flatten()
        self.mu = values.mean()
        
        losses = []
        for epoch in range(n_epochs):
            # Shuffle
            idx = np.random.permutation(len(rows))
            total_loss = 0.0
            
            for k in idx:
                u, i = rows[k], cols[k]
                r_true = values[k]
                
                # Prediction
                r_pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                error = r_true - r_pred
                
                # SGD updates
                self.b_u[u] += self.lr * (error - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (error - self.reg * self.b_i[i])
                
                P_u_old = self.P[u].copy()
                self.P[u] += self.lr * (error * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (error * P_u_old - self.reg * self.Q[i])
                
                total_loss += error ** 2
            
            rmse = np.sqrt(total_loss / len(rows))
            losses.append(rmse)
            if verbose and (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}, RMSE: {rmse:.4f}")
        
        return losses
    
    def predict(self, user_id, item_id):
        return self.mu + self.b_u[user_id] + self.b_i[item_id] + self.P[user_id] @ self.Q[item_id]
    
    def predict_all(self, user_id):
        return self.mu + self.b_u[user_id] + self.b_i + self.P[user_id] @ self.Q.T


# Train MF
print("Training Matrix Factorization (SGD)...")
mf = MatrixFactorization(n_users, n_items, n_factors=20, lr=0.005, reg=0.02)
losses = mf.fit(R, n_epochs=50)

# Plot training curve
plt.figure(figsize=(8, 5))
plt.plot(losses, linewidth=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('Matrix Factorization Training Convergence')
plt.show()

## 4. ALS (Alternating Least Squares)

Instead of SGD, alternate between fixing $P$ and solving for $Q$ (and vice versa). Each sub-problem is a least-squares problem with a closed-form solution:

Fix $Q$, update $P$: for each user $u$:
$$
\mathbf{p}_u = (Q_{I_u}^T Q_{I_u} + \lambda I)^{-1} Q_{I_u}^T \mathbf{r}_{u, I_u}
$$

where $I_u$ is the set of items rated by user $u$.

> **💡 Concept:** ALS is preferred for implicit feedback (Hu et al., 2008) and large-scale systems because it parallelizes well - each user's update is independent.

In [ ]:
class ALS:
    def __init__(self, n_users, n_items, n_factors=20, reg=0.1):
        self.n_factors = n_factors
        self.reg = reg
        self.P = np.random.randn(n_users, n_factors) * 0.1
        self.Q = np.random.randn(n_items, n_factors) * 0.1
    
    def fit(self, R, n_epochs=20, verbose=True):
        R_dense = R.toarray()
        mask = (R_dense != 0).astype(float)
        losses = []
        
        for epoch in range(n_epochs):
            # Update P (fix Q)
            for u in range(R.shape[0]):
                rated = np.where(mask[u] > 0)[0]
                if len(rated) == 0:
                    continue
                Q_sub = self.Q[rated]  # (n_rated, k)
                r_sub = R_dense[u, rated]  # (n_rated,)
                A = Q_sub.T @ Q_sub + self.reg * np.eye(self.n_factors)
                b = Q_sub.T @ r_sub
                self.P[u] = np.linalg.solve(A, b)
            
            # Update Q (fix P)
            for i in range(R.shape[1]):
                raters = np.where(mask[:, i] > 0)[0]
                if len(raters) == 0:
                    continue
                P_sub = self.P[raters]
                r_sub = R_dense[raters, i]
                A = P_sub.T @ P_sub + self.reg * np.eye(self.n_factors)
                b = P_sub.T @ r_sub
                self.Q[i] = np.linalg.solve(A, b)
            
            # Compute loss
            pred = self.P @ self.Q.T
            error = (R_dense - pred) * mask
            rmse = np.sqrt(np.sum(error ** 2) / mask.sum())
            losses.append(rmse)
            if verbose and (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1}/{n_epochs}, RMSE: {rmse:.4f}")
        
        return losses
    
    def predict_all(self, user_id):
        return self.P[user_id] @ self.Q.T


print("Training ALS...")
als = ALS(n_users, n_items, n_factors=20, reg=0.1)
als_losses = als.fit(R, n_epochs=20)

## 5. BPR: Bayesian Personalized Ranking

BPR optimizes for **pairwise ranking** rather than rating prediction:

$$
\mathcal{L}_{\text{BPR}} = -\sum_{(u, i, j) \in D_s} \ln \sigma(\hat{r}_{ui} - \hat{r}_{uj}) + \lambda \|\Theta\|^2
$$

where $(u, i, j)$ means user $u$ prefers item $i$ over item $j$, and $\sigma$ is the sigmoid function.

> **💡 Concept:** BPR is designed for implicit feedback. Instead of predicting exact ratings, it learns to rank interacted items above non-interacted items. This is more aligned with the actual recommendation task.

> **⚠️ Common Pitfall:** Negative sampling in BPR matters a lot. Uniform random negatives are easy but biased. Popular items that a user hasn't interacted with are "harder" negatives and lead to better learning.

In [ ]:
class MF_BPR(nn.Module):
    """Matrix Factorization with BPR loss in PyTorch."""
    
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, n_factors)
        self.item_emb = nn.Embedding(n_items, n_factors)
        
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
    
    def forward(self, user, item_pos, item_neg):
        u = self.user_emb(user)       # (batch, k)
        i_pos = self.item_emb(item_pos)  # (batch, k)
        i_neg = self.item_emb(item_neg)  # (batch, k)
        
        pos_score = (u * i_pos).sum(dim=1)  # (batch,)
        neg_score = (u * i_neg).sum(dim=1)  # (batch,)
        
        return pos_score, neg_score
    
    def predict(self, user):
        u = self.user_emb(user)
        scores = u @ self.item_emb.weight.T
        return scores


def bpr_loss(pos_score, neg_score):
    return -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-10).mean()


def sample_bpr_triplets(R, n_samples=1000):
    """Sample (user, positive_item, negative_item) triplets."""
    rows, cols = R.nonzero()
    n_items = R.shape[1]
    
    # Build user -> positive items mapping
    user_items = defaultdict(set)
    for u, i in zip(rows, cols):
        user_items[u].add(i)
    
    users, pos_items, neg_items = [], [], []
    for _ in range(n_samples):
        idx = np.random.randint(len(rows))
        u = rows[idx]
        i = cols[idx]
        # Sample negative
        j = np.random.randint(n_items)
        while j in user_items[u]:
            j = np.random.randint(n_items)
        users.append(u)
        pos_items.append(i)
        neg_items.append(j)
    
    return (torch.LongTensor(users), 
            torch.LongTensor(pos_items), 
            torch.LongTensor(neg_items))


# Train BPR-MF
from collections import defaultdict

torch.manual_seed(42)
model = MF_BPR(n_users, n_items, n_factors=32)
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)

bpr_losses = []
print("Training MF with BPR loss...")
for epoch in range(50):
    users, pos, neg = sample_bpr_triplets(R, n_samples=5000)
    
    pos_score, neg_score = model(users, pos, neg)
    loss = bpr_loss(pos_score, neg_score)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    bpr_losses.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}, BPR Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 5))
plt.plot(bpr_losses, linewidth=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('BPR-MF Training Convergence')
plt.show()

## 6. Comparison of Methods

Let's compare all four approaches on our synthetic data.

> **🔑 Pro Tip:** In practice, model-based methods (MF, BPR) almost always outperform memory-based methods (UserCF, ItemCF) on accuracy. But memory-based methods have advantages in interpretability and simplicity.

In [ ]:
# Evaluate all methods using ranking metrics
def evaluate_ranking(model_predict_fn, R, true_scores, k=10, n_eval_users=50):
    """Evaluate a recommender using NDCG@K and HitRate@K."""
    ndcgs, hit_rates = [], []
    
    eval_users = np.random.choice(n_users, n_eval_users, replace=False)
    
    for u in eval_users:
        rated = set(R[u].nonzero()[1])
        if len(rated) < 2:
            continue
        
        # Get predictions
        scores = model_predict_fn(u)
        # Mask rated items
        scores[list(rated)] = -np.inf
        
        # Top-K predicted items
        top_k = np.argsort(scores)[-k:][::-1]
        
        # "Relevant" items: those with true score above median
        threshold = np.median(true_scores[u])
        relevant = set(np.where(true_scores[u] > threshold)[0]) - rated
        
        if len(relevant) == 0:
            continue
        
        # NDCG
        rels = [1 if item in relevant else 0 for item in top_k]
        dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rels))
        ideal_rels = sorted(rels, reverse=True)
        idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal_rels))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
        
        # HitRate
        hit_rates.append(1.0 if len(set(top_k) & relevant) > 0 else 0.0)
    
    return np.mean(ndcgs), np.mean(hit_rates)


# Evaluate
np.random.seed(42)
methods = {
    'UserCF (Cosine)': user_cf_cosine.predict_all,
    'UserCF (Pearson)': user_cf_pearson.predict_all,
    'ItemCF': item_cf.predict_all,
    'MF (SGD)': mf.predict_all,
    'MF (ALS)': als.predict_all,
    'MF (BPR)': lambda u: model.predict(torch.LongTensor([u])).detach().numpy().flatten(),
}

results = {}
for name, predict_fn in methods.items():
    ndcg, hr = evaluate_ranking(predict_fn, R, true_scores, k=10)
    results[name] = {'NDCG@10': ndcg, 'HitRate@10': hr}
    print(f"{name:<20} NDCG@10={ndcg:.4f}  HitRate@10={hr:.4f}")

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
ndcg_vals = [results[n]['NDCG@10'] for n in names]
hr_vals = [results[n]['HitRate@10'] for n in names]

colors = ['#4CAF50', '#66BB6A', '#2196F3', '#FF9800', '#FFC107', '#E91E63']

axes[0].barh(names, ndcg_vals, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_xlabel('NDCG@10')
axes[0].set_title('NDCG@10 Comparison')

axes[1].barh(names, hr_vals, color=colors, edgecolor='black', alpha=0.8)
axes[1].set_xlabel('HitRate@10')
axes[1].set_title('HitRate@10 Comparison')

plt.tight_layout()
plt.show()

---

## Exercises

### 🏋️ Exercise 1: Implement MF with BPR from Scratch in PyTorch

Extend the BPR-MF model with user and item biases, and add L2 regularization explicitly.

In [ ]:
class MF_BPR_Extended(nn.Module):
    """MF-BPR with biases and explicit regularization."""
    
    def __init__(self, n_users, n_items, n_factors=32):
        super().__init__()
        # TODO: Add embeddings for users and items
        # TODO: Add bias parameters for users and items
        # TODO: Add global bias
        pass
    
    def forward(self, user, item_pos, item_neg):
        # TODO: Compute positive and negative scores with biases
        # score = global_bias + user_bias + item_bias + <user_emb, item_emb>
        pass
    
    def reg_loss(self, user, item_pos, item_neg):
        # TODO: Compute L2 regularization loss
        pass


# TODO: Train the extended model and compare with the basic version
# Plot both loss curves on the same figure

### 🏋️ Exercise 2: Similarity Measure Comparison

Implement Jaccard similarity and adjusted cosine similarity, then compare all four similarity measures.

In [ ]:
def jaccard_similarity(R):
    """
    Compute Jaccard similarity between users.
    Jaccard(u, v) = |I_u intersect I_v| / |I_u union I_v|
    """
    # TODO: Implement Jaccard similarity
    pass


def adjusted_cosine_similarity(R):
    """
    Adjusted cosine similarity (subtract item mean instead of user mean).
    Used in item-based CF to account for different rating scales.
    """
    # TODO: Implement adjusted cosine similarity
    pass


# TODO: Compare all four similarities on a subset of users
# Visualize the similarity matrices as heatmaps

### 🏋️ Exercise 3: Effect of Latent Dimension on MF

Train MF models with different numbers of latent factors and analyze the bias-variance tradeoff.

In [ ]:
# TODO: 
# 1. Train MF models with n_factors in [2, 5, 10, 20, 50, 100]
# 2. For each, record final training RMSE
# 3. Also evaluate on held-out entries (create a simple train/test split)
# 4. Plot training RMSE and test RMSE vs n_factors
# 5. Identify the "sweet spot" and explain overfitting behavior
pass

## Summary

In this notebook, we covered:

1. **User-based CF**: Find similar users via cosine/Pearson similarity, predict from their ratings
2. **Item-based CF**: Find similar items, more stable and scalable than UserCF
3. **Matrix Factorization**: Learn latent factors via SGD or ALS, the foundation of modern RecSys
4. **BPR Loss**: Pairwise ranking loss for implicit feedback, better aligned with recommendation task

### Key Takeaways

- Memory-based methods (UserCF, ItemCF) are simple but don't scale well
- MF learns generalizable latent factors and handles sparsity better
- BPR shifts the objective from rating prediction to ranking, which is often what we actually care about
- The number of latent factors is a key hyperparameter controlling the bias-variance tradeoff

### Next Up

In **Chapter 1.4**, we'll explore embeddings - the representation learning perspective that bridges classical MF and deep learning.